# importing libraries

In [ ]:
%run initial_setup.py

In [ ]:
import pandas as pd
import MySQLdb
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
from math import radians, cos, sin, asin, sqrt
import pymysql
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 100000)
import json
import urllib.request



# creating engines and connections

In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py


db = client.get_database('telematics_db')





# setting right pd options

In [ ]:
pd.set_option("display.max_rows",5)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# defining basic functions

In [ ]:
getTime=lambda x: pd.Timestamp(x).timestamp()
getdate=lambda x: tm.strftime('%d-%m-%Y', tm.localtime(x))
getDay=lambda x: tm.strftime('%Y-%m-%d', tm.localtime(x))
dttoday=int(tm.time()-tm.time()%86400-19800)




In [ ]:
def haversine(L1,L2):
    """
    Calculate the great circle distance between two points
    on the earth (specified in decimal degrees)
    """

    lon1=L1[0]
    lat1=L1[1]
    lon2=L2[0]
    lat2=L2[1]
    #print(lon1,lat1,lon2,lat2)
    # convert decimal degrees to radians
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])

    # haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    r = 6371000 # Radius of earth in kilometers. Use 3956 for miles
    return c * r




In [ ]:
def split(start,end,vehicleList1):
    l=[]
    threshhold=86400
    for vid in vehicleList1:
        while(start+threshhold<end):
            l.append([vid,start,start+threshhold])
            start+=threshhold
        l.append([vid,int(start),int(end)])

    return l

In [ ]:
def analysis(l):
    vid=l[0]
    day_start=l[1]
    day_end=l[2]
    print('day start is '+str(day_start))
    
    try:
        
        
        ######### USING MONGO ############
        #query={"vehicleId":int(vid) ,"time": {"$gte":day_start,"$lte":day_end}}
        #df=pd.DataFrame(list(db.gpsHistory.find(query,{"vehicleId":1, "location":1,"time":1,"speed":1,'created':1,"chargeOn":1,"usedSat":1,'gsmPct':1})))
        
        
        
        ########### USING API ###########
        url=GPS_API_BASE+"/gps/vehicles?vehicleIds={}&fromTime={}&toTime={}&sortOrder=&timeCondition=".format(vid,int(day_start),int(day_end))
        data=json.load(urllib.request.urlopen(url))
        print(url)
        df=pd.DataFrame(data['data'][str(vid)])
        
        
    except Exception as e:
        
        print("QUERY ISSUE AAYA HAI  " + str(e))
        return  pd.DataFrame() 
       
    
    if len(df)>10:
        
        print('the number of pings= '+str(len(df)))
        df=df.sort_values('time',ascending=False)
        
        df.reset_index(drop=True,inplace=True)
        no_info_distance_travelled=0
        
        try:
            
            device_history=pd.DataFrame(list(db.deviceHistory.find({'vehicleId':vid,'fromTime':{'$lte':day_end},'toTime':{'$gte':day_start}})))
            device_history.sort_values('fromTime',ascending=True,inplace=True)

            for b,brow in device_history.iterrows():
                no_info_start=list(filter(lambda i : i<brow['fromTime'],df.time))[-1]-5
                no_info_end=list(filter(lambda i : i>brow['toTime'],df.time))[0]+5

                no_info_distance_travelled+=df[df['time']>=no_info_start][df['time']<=no_info_end]['distance'].sum()
        except:
            no_info_distance_travelled=None
            print("NO NO_INFO")

            
        
        
        
        
        
        
        
        total_counts=len(df)
        
           
            
        if('usedSat' in df):
            sat_used_values=df['usedSat'].describe(percentiles=[0.01,0.05,0.1]).values[[1,4,5,6,7]]
            #['mean', '1%', '5%', '10%', '50%']]
                
        else:
            sat_used_values=[None]*5
            
            
            
        if('gsmPct' in df):
            gsm_pct_values=df['gsmPct'].describe(percentiles=[0.01,0.05,0.1]).values[[1,4,5,6,7]]
            #['mean', '1%', '5%', '10%', '50%']
        else:
            gsm_pct_values=[None]*5
                    
        print("pooooooooo")    
                
        df.loc[:,'loc2']=df['location'].shift(1)
        df.loc[:,'time1']=df['time'].shift(1)
            
        df=df[1:]
        
        time_diff_cons=df['time1']-df['time']
            
        df.loc[:,'time_diff_cons_pings']=time_diff_cons
        
        time_diff_cons_pings_values=df.time_diff_cons_pings.describe(percentiles=[0.80,0.95,0.99]).values[[1,4,5,6,7]]
        #'mean', '50%', '80%', '95%', '99%'
            
        ############################################################################# change later on
        
            
        #df.loc[:,'server_created_time']=df['created'].apply(getTime)
        
        df.loc[:,'server_created_time']=df['created']
        df['server_created_time']=df['server_created_time']/1000
        ###################################################
            
        df.loc[:,'server_delay']=df['server_created_time']-df['time']
        
        server_delay_values=df.server_delay.describe(percentiles=[0.80,0.95,0.99]).values[[1,4,5,6,7]]

        
        loc_list=df[['location','loc2']].values.tolist()

        dis=[]
        for loc in loc_list:
            y=haversine(loc[0],loc[1])
            dis.append(y)
            
        df['distance']=dis
        
        jump_values=df[df['distance']>500]['distance'].describe(percentiles=[0.80,0.95,0.99]).values[[0,1,4,5,6,7]]
        
        

        df.loc[:,'speed_self_cal']=df['distance']/df['time_diff_cons_pings']    
        df.loc[:,'moving_status']=df.apply(lambda row: 'running' if row['speed_self_cal']>0.8 else  'stop' ,axis=1)
        
        driving_count=len(df[df['moving_status']=='running'])
        
        stop_count=len(df[df['moving_status']=='stop'])
        
                                
        driving_time=df[df['moving_status']=='running']['time_diff_cons_pings'].sum()
            
        stop_time=df[df['moving_status']=='stop']['time_diff_cons_pings'].sum()
            
        day=getDay((df['time'].min()+df['time'].max())/2)

        total_distace_travelled=df['distance'].sum()
        
        
        df=df.sort_values('server_created_time',ascending=False)

        df.reset_index(drop=True,inplace=True)


        df['created_time_shift']=df['server_created_time'].shift(1)
        df.at[0,'created_time_shift']=df.iloc[0]['server_created_time']
            

        df['created_cons_time_diff']=df['created_time_shift']-df['server_created_time']


        live_time=df[df['created_cons_time_diff']<180]['created_cons_time_diff'].sum()
        offline_time=df[df['created_cons_time_diff']>180]['created_cons_time_diff'].sum()
        
        created_cons_time_diff_values=df[df['created_cons_time_diff']>180]['created_cons_time_diff'].describe(percentiles=[0.80,0.95,0.99]).values[[0,1,4,5,6,7]]
        
        
        if (df.time.min()-day_start)>=180:
            offline_time=offline_time+df.time.min()-day_start
        else:
            live_time=live_time+df.time.min()-day_start
        
        
        if (day_end-df.time.max())>=180:
            offline_time=offline_time+day_end-df.time.max()
        else :
            live_time=live_time+day_end-df.time.max()
            
        
        jump_distance=df[df['distance']>500]['distance'].sum()
        jump_percent=100*jump_distance/total_distace_travelled
        
        expected_run_pings_percent=1000*driving_count/driving_time
        expected_stop_pings_percent=3000*stop_count/stop_time
        
        
        no_info_duration=df[df['created_cons_time_diff']>3600]['created_cons_time_diff'].sum()
        
        
        if (df.time.min()-day_start)>=3600:
            no_info_duration=no_info_duration+df.time.min()-day_start
        
        elif (day_end-df.time.max())>=3600:
            no_info_duration=no_info_duration+day_end-df.time.max()
        
        
        

        try: 
            
            final_values=[vid,day,total_counts]+list(sat_used_values)+list(gsm_pct_values)+list(time_diff_cons_pings_values)+list(server_delay_values)+list(jump_values)+[driving_count,stop_count,driving_time,stop_time,total_distace_travelled,live_time,offline_time]+list(created_cons_time_diff_values)+[jump_distance,jump_percent,expected_run_pings_percent,expected_stop_pings_percent,no_info_distance_travelled,no_info_duration]
        except Exception as e:
            print(str(e))
        #print(final_values[:-2])
        print("KUCH TOH CHAL RAHA HAI")
        print(final_values)

        
        final_column_names=['vehicle_id','day','total_raw_pings',
                            'usedSat_mean', 'usedSat_pctl_1%', 'usedSat_pctl_5%', 'usedSat_pctl_10%', 'usedSat_pctl_50%',
                           'Gsm_mean', 'Gsm_pctl_1%', 'Gsm_pctl_5%', 'Gsm_pctl_10%', 'Gsm_pctl_50%',
                           'cnsctv_time_diff_mean', 'cnsctv_time_diff_50%', 'cnsctv_time_diff_80%', 'cnsctv_time_diff_95%', 'cnsctv_time_diff_99%',
                           'server_delay_mean', 'server_delay_50%', 'server_delay_80%', 'server_delay_95%', 'server_delay_99%',
                           'jump_ping_500mts_count','jump_500mts_mean', 'jump_500mts_50%', 'jump_500mts_80%', 'jump_500mts_95%', 'jump_500mts_99%',
                           'driving_pings','stop_pings','driving_time','stop_time','total_distace_travelled','live_time','offline_time',
                           'created_3mins_ping_count','created_cnsctv_mean', 'created_cnsctv_50%', 'created_cnsctv_80%', 'created_cnsctv_95%', 'created_cnsctv_99%',
                           'jump_distance','jump_percent','expected_run_pings_percent','expected_stop_pings_percent','no_info_distance_travelled','no_info_duration']


        day_wise=pd.DataFrame([final_values],columns=final_column_names)
        print(day_wise)
        
        return day_wise
    
                   
    else:
        print("DATA NAHI AANE WAALA")
        return pd.DataFrame()
            
        
        
        
        
        


# collecting data 

In [ ]:
aggregate=pd.read_sql("select * from operator_vehicles where vehicle_id=100002 and deboarding_time is null;",mysql_vds).rename(index=str,columns={"vehicle_id":"vehicleId"})

In [ ]:
#getting old data from metabse 
old=pd.read_sql("select * from device_performance limit 10",con=engine)

In [ ]:
last_date_algo_ran=old.date.max()
start=getTime(last_date_algo_ran)-19800+86400
end=getTime(dt.date.today())-19800+86400


In [ ]:
# vehicleList=aggregate.vehicleId.tolist()
############################### FOR THE OPS TEAM ANALYSIS #############################
vehicleList=[100002,100002,100002,100002,100002,100002,100002,100002]
vid_start_end=split(start,end,vehicleList)


In [ ]:
print("Total {} iterations to go".format(len(vid_start_end)))


In [ ]:
from multiprocessing.dummy import Pool as ThreadPool
pool = ThreadPool(4)
results=pool.map(analysis,vid_start_end)
#results=map(analysis,vid_start_end)
pool.close()
pool.join()

data=pd.concat(results)
        
        
        

In [ ]:
data.to_csv("bala_ji_data.csv")

In [ ]:
d=pd.DataFrame(data['data']['100002'])


In [ ]:
        query={"vehicleId":100002}
        df=pd.DataFrame(list(db.gpsHistory.find(query,{"vehicleId":1, "location":1,"time":1,"speed":1,'created':1,"chargeOn":1,"usedSat":1,'gsmPct':1}).limit(100)))


In [ ]:
df['usedSat'].describe(percentiles=[0.80,0.95,0.99]).keys()[[0,1,4,5,6,7]]

In [ ]:
%run inital_setup.py

In [ ]:
data

In [ ]:
pd.read_sql("select * from vehicles",mysql_cn)

In [ ]:
timestamp_ToDateTime(222)

In [ ]:
gettime?

In [ ]:
df

In [ ]:
from bisect import bisect_right,bisect_left

In [ ]:
a=pd.Series([2,3,33,4,5,8,9,7,75,5,3,7,543])

In [ ]:
a

In [ ]:
a[bisect_left(a,88)]

In [ ]:
a[bisect_right(a,88)]

In [ ]:
res = list(filter(lambda i: i > 0.6, a))[0] 


In [ ]:
list(filter(lambda i : i<6,a))